In [1]:
from torchvision.models import (
    MobileNet_V3_Small_Weights,
    mobilenet_v3_small
)




In [2]:
dir(MobileNet_V3_Small_Weights.IMAGENET1K_V1)

['__class__',
 '__doc__',
 '__eq__',
 '__hash__',
 '__module__',
 'get_state_dict',
 'meta',
 'name',
 'transforms',
 'url',
 'value',
 'verify']

In [3]:
model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1.value)
model.eval()

/Users/smsantho/Development/ora_proj/external/vision-inference-bench/.venv/lib/python3.14/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Small_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(16, 16

In [4]:
preprocess = MobileNet_V3_Small_Weights.IMAGENET1K_V1.transforms()

In [5]:
from PIL import Image

image = 0

with Image.open("/Users/smsantho/Downloads/camel.jpg") as img:
    image = img.convert("RGB")

image = preprocess(image)
image.size()

batch = image.unsqueeze(0)
batch.size()

torch.Size([1, 3, 224, 224])

In [6]:
from torch import torch

with torch.inference_mode():

    logits = model(batch)
    probabilities = logits.softmax(dim=1)

    print(MobileNet_V3_Small_Weights.IMAGENET1K_V1.meta['categories'][probabilities.argmax()])
    print(f"{probabilities[0][probabilities.argmax()]*100} %")

    # top 5

    for i in probabilities[0].topk(5).indices: 
        cat = MobileNet_V3_Small_Weights.IMAGENET1K_V1.meta['categories'][i]
        prob = probabilities[0][i]
        print(f"{cat} => {prob*100}%")

Arabian camel
99.96979522705078 %
Arabian camel => 99.96979522705078%
llama => 0.016512814909219742%
sorrel => 0.006010472308844328%
hartebeest => 0.0038017339538782835%
bighorn => 0.0013933863956481218%


In [7]:
from pathlib import Path

root = Path("/tmp/onnx")
root.mkdir(parents=True, exist_ok=True)

fp32_path = root / "mobilenet_fp32.onnx"

In [8]:

# export to onnx

torch.onnx.export(
    model,
    (batch),
    fp32_path,
    dynamo=True, 
    external_data=False,    
    input_names=["input"],
    output_names=["logit"]
)

[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 20},
            producer_name='pytorch',
            producer_version='2.14.1',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[1,3,224,224]>
            ),
            outputs=(
                %"logit"<FLOAT,[1,1000]>
            ),
            initializers=(
                %"features.0.0.weight"<FLOAT,[16,3,3,3]>{Tensor(...)},
                %"features.1.block.0.0.weight"<FLOAT,[16,1,3,3]>{Tensor(...)},
                %"features.1.block.1.fc1.weight"<FLOAT,[8,16,1,1]>{TorchTensor(...)},
                %"features.1.block.1.fc1.bias"<FLOAT,[8]>{TorchTensor<FLOAT,[8]>(Parameter containing: tensor([-1.7194e-09, -2.3166e-09, -1.6355e-04, -1.8775e-09, -1.2891e-09, 1.7532e-08, -5.6571e-09,  9.6723e-11], requires_grad=True), name='features.1.block.1.fc1.bias')},
                %"f

In [9]:
from onnxruntime.quantization.shape_inference import quant_pre_process

prepros_qfp32_path = root / "pre_quant_mobilenet_fp32.onnx"

quant_pre_process(
    fp32_path,
    prepros_qfp32_path
)

In [10]:
# calibration data reader
from onnxruntime.quantization import CalibrationDataReader
from PIL import Image

def loadPath(path):
    with Image.open(path) as img:
        return preprocess(img.convert("RGB")).unsqueeze(0).numpy()

class ImageCallibrationDataReader(CalibrationDataReader):

    def __init__(self,paths):
        self.paths = iter(paths)

    def get_next(self):

        path = next(self.paths, None)

        if path is None: return None

        return {
            "input": loadPath(path) 
        }
        

In [11]:
# downlaoded some images to calibration folder from from each of Imagenette’s ten classes

from onnxruntime.quantization import (
    quantize_static,
    QuantFormat,
    QuantType,
    CalibrationMethod
)


outputPath = root / "q_output.onnx"

paths = list((root / "calibration").glob("*.jpg"))

# (model_input: str | Path | ModelProto, model_output: str | Path, calibration_data_reader: CalibrationDataReader | None = None, quant_format: QuantFormat = QuantFormat.QDQ, op_types_to_quantize: Any | None = None, per_channel: bool = False, reduce_range: bool = False, activation_type: QuantType = QuantType.QInt8, weight_type: QuantType = QuantType.QInt8, nodes_to_quantize: Any | None = None, nodes_to_exclude: Any | None = None, use_external_data_format: bool = False, calibrate_method: CalibrationMethod = CalibrationMethod.MinMax, calibration_providers: Any | None = None, extra_options: Any | None = None, calibration_cache_path: str | Path | None = None) -> None
quantize_static(
    prepros_qfp32_path,
    outputPath,
    ImageCallibrationDataReader(paths),
    QuantFormat.QDQ,
    ["Gemm","Conv","MatMul"],
    activation_type=QuantType.QInt8,
    weight_type=QuantType.QInt8,
    per_channel=True,
    calibrate_method=CalibrationMethod.MinMax
)



In [12]:
import onnxruntime as ort

options = ort.SessionOptions()
options.intra_op_num_threads = 1  # One thread within an operation.
options.inter_op_num_threads = 1

fp32_session = ort.InferenceSession(
    str(fp32_path),
    sess_options=options,
    providers=["CPUExecutionProvider"],
)

int8_session = ort.InferenceSession(
    str(outputPath),
    sess_options=options,
    providers=["CPUExecutionProvider"],
)

input_array = batch.numpy()  # Prepare the input before timing anything.

In [13]:
import time
import numpy as np

def measure_latency(run, warmup=20, repeats=200):
    for _ in range(warmup):
        run()  # Allow initial setup and caches to settle.

    durations_ms = []

    for _ in range(repeats):
        start = time.perf_counter_ns()
        run()
        elapsed_ms = (time.perf_counter_ns() - start) / 1_000_000
        durations_ms.append(elapsed_ms)

    return {
        "median_ms": float(np.median(durations_ms)),
        "p95_ms": float(np.percentile(durations_ms, 95)),
    }


fp32_latency = measure_latency(
    lambda: fp32_session.run(["logit"], {"input": input_array})
)

int8_latency = measure_latency(
    lambda: int8_session.run(["logit"], {"input": input_array})
)

print(fp32_latency, int8_latency)

{'median_ms': 3.0949999999999998, 'p95_ms': 3.29958155} {'median_ms': 1.4950625, 'p95_ms': 1.60251875}


In [14]:
# Model sizes, single-image output comparison, and benchmark conditions.
import json
import platform
import subprocess
import sys
import torch
import torchvision

fp32_output = fp32_session.run(
    None, {fp32_session.get_inputs()[0].name: input_array}
)[0]
int8_output = int8_session.run(
    None, {int8_session.get_inputs()[0].name: input_array}
)[0]

if not (np.isfinite(fp32_output).all() and np.isfinite(int8_output).all()):
    raise ValueError("Model outputs contain non-finite values")

difference = np.abs(fp32_output - int8_output)
cpu_model = platform.processor()
if platform.system() == "Darwin":
    cpu_model = subprocess.run(
        ["sysctl", "-n", "machdep.cpu.brand_string"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()

results = {
    "fp32": {**fp32_latency, "file_bytes": fp32_path.stat().st_size},
    "int8": {**int8_latency, "file_bytes": outputPath.stat().st_size},
    "median_speedup": fp32_latency["median_ms"] / int8_latency["median_ms"],
    "file_size_reduction_pct": 100 * (
        1 - outputPath.stat().st_size / fp32_path.stat().st_size
    ),
    "single_image_comparison": {
        "max_absolute_logit_difference": float(difference.max()),
        "mean_absolute_logit_difference": float(difference.mean()),
        "fp32_top1": int(fp32_output[0].argmax()),
        "int8_top1": int(int8_output[0].argmax()),
        "top1_agreement": bool(fp32_output[0].argmax() == int8_output[0].argmax()),
    },
    "conditions": {
        "cpu_model": cpu_model,
        "os": platform.platform(),
        "architecture": platform.machine(),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "onnxruntime": ort.__version__,
        "fp32_model_path": str(fp32_path),
        "int8_model_path": str(outputPath),
        "fp32_providers": fp32_session.get_providers(),
        "int8_providers": int8_session.get_providers(),
        "input_shape": list(input_array.shape),
        "input_dtype": str(input_array.dtype),
        "fp32_intra_op_threads": fp32_session.get_session_options().intra_op_num_threads,
        "int8_intra_op_threads": int8_session.get_session_options().intra_op_num_threads,
        "fp32_inter_op_threads": fp32_session.get_session_options().inter_op_num_threads,
        "int8_inter_op_threads": int8_session.get_session_options().inter_op_num_threads,
        "fp32_execution_mode": str(fp32_session.get_session_options().execution_mode),
        "int8_execution_mode": str(int8_session.get_session_options().execution_mode),
        "warmup": 20,
        "repeats": 200,
        "timing_scope": "session.run on one preprocessed image; excludes loading and preprocessing",
        "thread_count_note": "0 means runtime default, not zero execution threads",
    },
    "accuracy": None,
    "memory": None,
}
print(json.dumps(results, indent=2))


{
  "fp32": {
    "median_ms": 3.0949999999999998,
    "p95_ms": 3.29958155,
    "file_bytes": 10475490
  },
  "int8": {
    "median_ms": 1.4950625,
    "p95_ms": 1.60251875,
    "file_bytes": 2820172
  },
  "median_speedup": 2.07014756908156,
  "file_size_reduction_pct": 73.07837628597802,
  "single_image_comparison": {
    "max_absolute_logit_difference": 21.43025016784668,
    "mean_absolute_logit_difference": 3.318148374557495,
    "fp32_top1": 354,
    "int8_top1": 723,
    "top1_agreement": false
  },
  "conditions": {
    "cpu_model": "Apple M4 Pro",
    "os": "macOS-26.7.1-arm64-arm-64bit-Mach-O",
    "architecture": "arm64",
    "python": "3.14.0",
    "torch": "2.14.1",
    "torchvision": "0.29.1",
    "onnxruntime": "1.31.0",
    "fp32_model_path": "/tmp/onnx/mobilenet_fp32.onnx",
    "int8_model_path": "/tmp/onnx/q_output.onnx",
    "fp32_providers": [
      "CPUExecutionProvider"
    ],
    "int8_providers": [
      "CPUExecutionProvider"
    ],
    "input_shape": [
      

In [15]:
# Held-out accuracy: add (image path, ImageNet class index 0-999) pairs here.
# Imagenette labels 0-9 must be mapped to ImageNet indices first.
eval_samples = []

def evaluate_accuracy(session, samples):
    if not samples:
        raise ValueError("Evaluation dataset is empty")

    input_name = session.get_inputs()[0].name
    calibration_paths = {path.resolve() for path in paths}
    correct = 0
    for image_path, imagenet_class_index in samples:
        if Path(image_path).resolve() in calibration_paths:
            raise ValueError("Use held-out images, not calibration images")
        if not isinstance(imagenet_class_index, int) or not 0 <= imagenet_class_index < 1000:
            raise ValueError("Labels must be ImageNet integer indices from 0 to 999")
        output = session.run(None, {input_name: loadPath(image_path)})[0]
        correct += int(output[0].argmax() == imagenet_class_index)

    return {"count": len(samples), "top1_accuracy_pct": 100 * correct / len(samples)}

results["accuracy"] = None
if eval_samples:
    fp32_accuracy = evaluate_accuracy(fp32_session, eval_samples)
    int8_accuracy = evaluate_accuracy(int8_session, eval_samples)
    results["accuracy"] = {
        "fp32": fp32_accuracy,
        "int8": int8_accuracy,
        "accuracy_drop_percentage_points": (
            fp32_accuracy["top1_accuracy_pct"] - int8_accuracy["top1_accuracy_pct"]
        ),
    }
    print(json.dumps(results["accuracy"], indent=2))
else:
    print("Accuracy not measured: populate eval_samples with separate labeled images.")


Accuracy not measured: populate eval_samples with separate labeled images.


In [16]:
# Peak whole-process RSS: each model runs in a fresh Python process.
# Includes Python, NumPy, ONNX Runtime, model loading, warm-up, and inference.
import tempfile

memory_worker = r"""
import json
import platform
import resource
import sys
import numpy as np
import onnxruntime as ort

array = np.load(sys.argv[2], allow_pickle=False)
options = ort.SessionOptions()
options.intra_op_num_threads = int(sys.argv[3])
options.inter_op_num_threads = int(sys.argv[4])
options.execution_mode = getattr(ort.ExecutionMode, sys.argv[5])
session = ort.InferenceSession(
    sys.argv[1], sess_options=options, providers=["CPUExecutionProvider"]
)
feed = {session.get_inputs()[0].name: array}
for _ in range(20 + 200):
    session.run(None, feed)

peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
# macOS reports bytes; Linux reports KiB.
peak_bytes = int(peak if platform.system() == "Darwin" else peak * 1024)
print(json.dumps({"peak_process_rss_bytes": peak_bytes, "peak_process_rss_mib": peak_bytes / 2**20}))
"""

if platform.system() not in {"Darwin", "Linux"}:
    raise RuntimeError("This RSS measurement supports macOS and Linux")

memory = {}
with tempfile.TemporaryDirectory() as temporary_directory:
    array_path = Path(temporary_directory) / "input.npy"
    np.save(array_path, input_array, allow_pickle=False)
    for name, model_path, session in [
        ("fp32", fp32_path, fp32_session),
        ("int8", outputPath, int8_session),
    ]:
        session_options = session.get_session_options()
        completed = subprocess.run(
            [sys.executable, "-c", memory_worker, str(model_path), str(array_path),
             str(session_options.intra_op_num_threads),
             str(session_options.inter_op_num_threads),
             session_options.execution_mode.name],
            capture_output=True, text=True,
        )
        if completed.returncode:
            raise RuntimeError(f"{name} memory measurement failed:\n{completed.stderr}")
        memory[name] = json.loads(completed.stdout)

results["memory"] = {
    "scope": "Peak whole-process RSS in a fresh process per model, including runtime overhead",
    **memory,
}
print(json.dumps(results["memory"], indent=2))


{
  "scope": "Peak whole-process RSS in a fresh process per model, including runtime overhead",
  "fp32": {
    "peak_process_rss_bytes": 89522176,
    "peak_process_rss_mib": 85.375
  },
  "int8": {
    "peak_process_rss_bytes": 72957952,
    "peak_process_rss_mib": 69.578125
  }
}


In [17]:
# Save the metrics collected above. Unmeasured accuracy/memory remain null.
results_path = root / "benchmark_results.json"
results_path.write_text(json.dumps(results, indent=2) + "\n")
print(f"Saved benchmark results to {results_path}")


Saved benchmark results to /tmp/onnx/benchmark_results.json
